# 00 · Inputs check

**Parts 7–8** read everything they need from the other parts and rebuild nothing. This notebook checks,
before any analysis, that each input exists, has the columns we rely on (the contract in
[`docs/INTERFACES.md`](docs/INTERFACES.md)), and matches what its owner reported.

It runs in a minute and writes only `data/analysis_panel.parquet`. Run it first whenever a teammate
says they updated their output.

| Input | Owner | File |
|---|---|---|
| events: surprise, returns, flags | parts 3–4 | `3_4_surprise_returns/data/events_final.parquet` |
| cleaned release texts | part 5 | `5_data cleaning/data/clean/clean_*.parquet` |
| model scores | part 6 (via part 5's cache) | `5_data cleaning/cache/model_outputs.sqlite` |
| sectors (SIC) | parts 3–4 | `3_4_surprise_returns/cache/crsp_names.parquet` |
| frozen holdout | part 5 | `5_data cleaning/data/holdout_ids.txt` |

In [1]:
# --- Setup (same in every notebook) ----------------------------------------
import os, sys, warnings
from pathlib import Path

try:                                   # on Colab: mount Drive and move to this folder
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/S08 NLP/7_8_baselines_evaluation")
except ImportError:                    # on a laptop: run Jupyter from this folder
    pass

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "code"))

# SYNTHETIC = True runs everything on fake data with a planted effect and writes to
# results/_synthetic/ (never over the real results). Set STEP78_SYNTHETIC=1 or edit here.
SYNTHETIC = os.environ.get("STEP78_SYNTHETIC", "0") == "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

import settings as S
if SYNTHETIC:
    S.redirect_outputs("synthetic")
S.ensure_dirs()
import io_inputs as IO, outputs as OUT, pipeline as P
print("TEAM_DIR :", S.TEAM_DIR)
print("outputs  :", S.RESULTS)
print("mode     :", "SYNTHETIC DEMO" if SYNTHETIC else "real data")

TEAM_DIR : /Users/maximelacroix/Downloads/S08 NLP
outputs  : /Users/maximelacroix/Downloads/S08 NLP/7_8_baselines_evaluation/results
mode     : real data


## 1. The event table (parts 3–4)

`load_events` asserts that every required column exists, that `event_id` is unique, and that the
event → 8-K link is one-to-one (one accession never serves two events). It converts nullable pandas
types to plain numpy types, which statsmodels and scikit-learn need.

The **version stamps** identify which build of parts 3–4 we are reading; they are written into
`logs/runs.jsonl` with every table we save.

In [2]:
ev, texts, sectors, scores, prov = P.load_all(SYNTHETIC)
print(f"{len(ev):,} events, {ev.shape[1]} columns")
OUT.upstream_versions(ev, texts)

texts: 111 cleaned documents from 1 year files in /Users/maximelacroix/Downloads/S08 NLP/5_data cleaning/data/clean
texts: 98 more from the fallback export release_texts.csv (not cleaned by part 5: use for development only)


sectors: 100.0% of events have a point-in-time SIC
model scores: the cache is empty (part 6 has not run)
217,611 events, 80 columns


{'surprise_version': 'surprise_v2_wire_first',
 'returns_version': 'returns_v3_placebo_day_m1',
 'sample_version': 'sample_v1',
 'clean_version': 'fallback,v1'}

**Cross-check with part 4.** The number of primary-sample events per split must equal part 4's
own `results/sample_by_split.csv`. If it does not, we are reading a different build of the event
table than the one part 4 reported on.

In [3]:
mine = ev.groupby("split")[["in_primary", "in_text_sample"]].sum().T
ref_path = S.PART34 / "results" / "sample_by_split.csv"
if ref_path.exists() and not SYNTHETIC:
    ref = pd.read_csv(ref_path, index_col=0)[mine.columns]
    display(pd.concat({"ours": mine, "part 4": ref.loc[mine.index]}))
    assert (mine.to_numpy() == ref.loc[mine.index].to_numpy()).all(), "sample counts differ from part 4"
    print("OK: identical to part 4's sample_by_split.csv")
else:
    display(mine)

dev  excluded_2020  holdout  validation
ours   in_primary      35508           4114    22145       11375
       in_text_sample  51529           5232    26503       14523
part 4 in_primary      35508           4114    22145       11375
       in_text_sample  51529           5232    26503       14523

OK: identical to part 4's sample_by_split.csv


## 2. Texts (part 5)

We read part 5's `text_body`: the release *without* safe-harbour, "About the company", contacts and
conference-call sections. Those sections repeat across a firm's releases and would let a bag-of-words
model recognise the firm instead of reading the news. Where part 5 has not cleaned a filing yet, part
2's older export is used and labelled `fallback_release_texts` (development only).

Coverage by year tells us which baselines can run: TF-IDF needs at least 500 training texts in each
fold, i.e. several cleaned years.

In [4]:
cov = (ev[ev["in_primary"]].assign(year=ev["reaction_date"].dt.year,
                                   has_text=ev["accession"].isin(texts["accession"]))
       .groupby("year")["has_text"].agg(events="size", with_text="sum"))
cov["share"] = cov["with_text"] / cov["events"]
print(texts["text_source"].value_counts().to_string())
cov.T

text_source
clean_v1                  111
fallback_release_texts     98


year,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
events,"2,714.0000","2,785.0000","3,009.0000","3,197.0000","3,256.0000","3,278.0000","3,238.0000","3,334.0000","3,370.0000","3,747.0000","3,580.0000","3,746.0000","3,757.0000","3,872.0000","4,114.0000","4,432.0000","4,606.0000","4,528.0000","4,298.0000","4,281.0000"
with_text,70.0000,1.0000,3.0000,4.0000,3.0000,5.0000,2.0000,4.0000,4.0000,4.0000,5.0000,4.0000,3.0000,3.0000,5.0000,5.0000,5.0000,4.0000,4.0000,2.0000
share,0.0258,0.0004,0.0010,0.0013,0.0009,0.0015,0.0006,0.0012,0.0012,0.0011,0.0014,0.0011,0.0008,0.0008,0.0012,0.0011,0.0011,0.0009,0.0009,0.0005


## 3. Model scores (part 6)

Part 6 writes every model answer into part 5's cache (`ModelCache`). Each (model, prompt, variant)
becomes one signal column here. The provenance table lists the model revisions and cleaning versions
behind each column: a signal mixing two revisions is a red flag.

In [5]:
print(f"{max(scores.shape[1] - 1, 0)} model signals")
prov

0 model signals


""


## 4. Loughran–McDonald dictionary and holdout status

In [6]:
from baseline_lm import find_dictionary
print("LM dictionary :", find_dictionary() or "MISSING: see resources/lm/README.md")
frozen = IO.load_frozen_holdout()
print("holdout frozen:", f"yes, {len(frozen):,} accessions" if frozen else "not yet (part 5: splits.py --freeze)")
print("UNLOCK_HOLDOUT:", os.environ.get("UNLOCK_HOLDOUT", "not set (2021-2025 hidden: correct until the freeze)"))

LM dictionary : MISSING: see resources/lm/README.md
holdout frozen: not yet (part 5: splits.py --freeze)
UNLOCK_HOLDOUT: not set (2021-2025 hidden: correct until the freeze)


## 5. The analysis panel

One row per event of the primary sample, with seasons, within-season SUE ranks and deciles,
winsorised outcomes, controls, FF12 sectors and text metadata. Built once and saved, so every
baseline and every test reads the same events (see `docs/METHODOLOGY.md`, section 3). The holdout
guard of part 3-4's `common.py` runs here.

In [7]:
panel = P.build_inputs(ev, texts, sectors)
panel.to_parquet(S.PANEL, index=False)
print("->", S.PANEL)
P.describe_panel(panel)

panel: 217,611 events -> 69,028 in sample 'in_primary' without 2020


holdout guard: dropped 22,145 rows from 2021-2025


-> /Users/maximelacroix/Downloads/S08 NLP/7_8_baselines_evaluation/data/analysis_panel.parquet


,events,with_text,seasons,first,last,median_sue,mean_car_0_1_ff4
split,,,,,,,
dev,35508,105,44,2006-01-04,2016-12-22,0.0005,0.0015
validation,11375,10,12,2017-01-04,2019-12-20,0.0006,0.0005


In [8]:
# Sector mix of the panel (FF12, point-in-time SIC)
(panel.groupby("split")["ff12"].value_counts(normalize=True).unstack(0) * 100).round(1)

split,dev,validation
ff12,,
BusEq,16.7000,14.7000
Chems,3.3000,3.1000
Durbl,2.5000,2.4000
Enrgy,4.9000,5.0000
Hlth,9.3000,9.8000
Manuf,10.5000,9.1000
Money,15.3000,17.0000
NoDur,4.8000,4.8000
Other,13.6000,15.0000
